# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VipS-2004/flyrank1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue ranks content using the Week-4 heuristic baseline score. Higher scores are reviewed first.

The main reason codes are:

- **STALE_CONTENT** — content is older than 365 days and should be considered for a refresh.
- **LOW_CTR** — content has a low click-through rate and may need a review of its title, search intent, or search-result presentation.
- **LOW_ENGAGEMENT** — content has weaker engagement signals and should be reviewed for relevance, clarity, and user experience.

The recommended action is currently **Refresh Content**. The score is a prioritization signal, not a guarantee that a page needs to be changed. A human should review the page and its context before taking action.

In [13]:
!git clone https://github.com/VipS-2004/flyrank1.git

fatal: destination path 'flyrank1' already exists and is not an empty directory.


In [14]:
import pandas as pd
import os


df = pd.read_csv(
    "/content/flyrank1/data/raw/content_refresh_anonymized.csv"
)


df["baseline_score"] = (
    0.4 * (df["content_age_days"] / df["content_age_days"].max())
    + 0.3 * (1 - df["ctr"] / 100)
    + 0.3 * (1 - df["engagement_rate"] / 100)
)


def reason(row):
    if row["content_age_days"] > 365:
        return "STALE_CONTENT"
    elif row["ctr"] < 2:
        return "LOW_CTR"
    else:
        return "LOW_ENGAGEMENT"

df["reason_code"] = df.apply(reason, axis=1)


df["action"] = "Refresh Content"


queue = df.sort_values(
    "baseline_score",
    ascending=False
)


display(
    queue[
        ["content_id", "baseline_score", "reason_code", "action"]
    ].head(20)
)


,content_id,baseline_score,reason_code,action
3604,content_5f7d77cf01e3,0.995035,STALE_CONTENT,Refresh Content
2115,content_a0c3be8b0794,0.995035,STALE_CONTENT,Refresh Content
7500,content_5b5e85993c2b,0.995035,STALE_CONTENT,Refresh Content
19810,content_9d17befb32b0,0.995035,STALE_CONTENT,Refresh Content
7397,content_bb5a87d3be4f,0.995035,STALE_CONTENT,Refresh Content
25831,content_5466a8258699,0.995035,STALE_CONTENT,Refresh Content
25956,content_7faeb2d774be,0.995035,STALE_CONTENT,Refresh Content
12934,content_70641aa29f3e,0.995035,STALE_CONTENT,Refresh Content
12972,content_d5b833d82e72,0.995035,STALE_CONTENT,Refresh Content
17338,content_8e4c8d698268,0.995035,STALE_CONTENT,Refresh Content


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help a content or SEO team prioritize pages for human review. It turns the baseline score and reason codes into a ranked queue so that reviewers can start with the highest-priority items.

The output is decision-support, not an automatic decision about whether content should be changed. A high score means an item should be reviewed earlier; it does not mean that refreshing the page will definitely improve performance.

### Limits

The current queue is based on a simple heuristic using content age, CTR, and engagement rate. It does not understand the actual quality, search intent, topic, business importance, or accuracy of a page.

The model and validation work also showed weaker results under a client-grouped split, so the recommendations should not be treated as proven to generalize to unseen clients.

The playbook is therefore intended for prioritization and discussion, not autonomous content changes or production decisions.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Before taking action on a recommendation, a person should review the actual page and confirm:

- The page is still relevant to the intended audience.
- The content is accurate and not outdated in substance.
- The search intent and topic are still appropriate.
- The recommendation is not based only on the score or reason code.
- Any proposed changes are consistent with the site's goals and content standards.

### No-go list

The system should NOT automatically:

- Rewrite or publish content.
- Delete or unpublish pages.
- Change important business or product information.
- Make SEO changes without human review.
- Treat a high score as proof that a page will improve after a refresh.
- Make irreversible changes based only on the ranking queue.

The queue is a prioritization tool. A human remains responsible for deciding whether and how to act.

In [16]:


no_go_actions = [
    "Automatic content publishing",
    "Automatic page deletion",
    "Automatic SEO changes",
    "Irreversible changes without human approval"
]

print("Human review required before action: YES")
print("\nNo-go actions:")
for action in no_go_actions:
    print("-", action)


Human review required before action: YES

No-go actions:
- Automatic content publishing
- Automatic page deletion
- Automatic SEO changes
- Irreversible changes without human approval


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The recommendations should be reviewed periodically because content performance and the underlying data can change over time.

I would review the playbook if:

- The distribution of content age, CTR, or engagement rate changes substantially.
- The reason-code distribution changes unexpectedly.
- The ranked queue becomes dominated by one reason code.
- The relationship between the selected signals and engagement changes.
- Validation performance gets materially worse on newer data.
- New content types or major changes in the content strategy are introduced.

A retraining or recalibration step should be considered when the input data or relationships have changed enough that the current recommendations are no longer representative.

Because this is a non-production prototype, these are review triggers rather than automated retraining rules. A person should inspect the changes before updating the model or heuristic.

In [17]:


reason_distribution = (
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

print("Reason-code distribution:")
display(reason_distribution)

print("Number of ranked items:", len(queue))
print("Unique reason codes:", queue["reason_code"].nunique())


Reason-code distribution:


,reason_code,count
0,LOW_CTR,22931
1,STALE_CONTENT,6360
2,LOW_ENGAGEMENT,709


Number of ranked items: 30000
Unique reason codes: 3


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exported outputs

The ranked queue is exported so the research paper can reuse the same action priorities and reason codes produced by this notebook.

The CSV is generated from the current dataset and ranking logic rather than being manually edited. This keeps the paper's recommendations traceable to the notebook.

The exported queue is saved under `work/outputs/` as required by the project structure.

In [18]:
import os

# Create the required output directory
output_dir = "/content/flyrank1/work/outputs"
os.makedirs(output_dir, exist_ok=True)

# Export the complete ranked queue
output_path = os.path.join(
    output_dir,
    "ranked_action_queue.csv"
)

queue[
    ["content_id", "baseline_score", "reason_code", "action"]
].to_csv(
    output_path,
    index=False
)

print("Exported queue:")
print(output_path)

print("\nRows exported:", len(queue))
print("Columns exported:")
print(list(queue[
    ["content_id", "baseline_score", "reason_code", "action"]
].columns))


Exported queue:
/content/flyrank1/work/outputs/ranked_action_queue.csv

Rows exported: 30000
Columns exported:
['content_id', 'baseline_score', 'reason_code', 'action']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.